In [ ]:
%pip install langsmith openai python-dotenv langchain langchain-openai langchain-pinecone

In [ ]:
%pip install -U langsmith langchain[openai] langchain-community

# 1. 환경변수 로딩 , 2. 임베딩 모델 선언

In [ ]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

# 3. Retriever 생성

In [ ]:
from langchain_pinecone import PineconeVectorStore

index_name ='tax-with-markdown' #'tax-table-index' #'tax-upstage-index'

database = PineconeVectorStore.from_existing_index(embedding = embedding, index_name = index_name)

retriever = database.as_retriever()

# 4. RAG-Bot 생성

In [ ]:
# 수정된 부분: from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage
from langsmith import traceable

# 수정된 부분: llm = ChatOpenAI(model="gpt-4.1", temperature=1)
llm = ChatUpstage(model="solar-pro3")

# Add decorator so this function is traced in LangSmith
@traceable()
def rag_bot(question: str) -> dict:
    # LangChain retriever will be automatically traced
    # NOTE: 위에서 선언한 retriever를 사용하여 VectorStore에 접근한 후 데이터를 가져옴(Retrieval).
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    # 수정된 부분: 질문 수정
    # NOTE: retriever를 통해 가져온 문서들을 system prompt로 전달
    instructions = f"""당신은 한국의 소득세 전문가입니다.
       아래 소득세법을 참고해서 사용자의 질문에 답변해주세요.

소득세법:
{docs_string}"""
    # langchain ChatModel will be automatically traced
    ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    # NOTE: 'Evaluators'를 사용해서 'answer'와 'documents'를 평가할 예정
    return {"answer": ai_msg.content, "documents": docs}

# 5.데이터 생성 

In [ ]:
from langsmith import Client

client = Client()

# 수정된 부분: 평가 데이터 셋
# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "제1조에 따른 소득세법의 목적은 무엇인가요?"},
        "outputs": {"answer": "소득세법의 목적은 소득의 성격과 납세자의 부담능력에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지하는 것입니다."},
        "metadata": {"contexts": "제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다."},
    },
    {
        "inputs": {"question": "'거주자'는 소득세법에서 어떻게 정의되나요?"},
        "outputs": {"answer": "'거주자'는 한국에 주소를 두거나 183일 이상 거소를 둔 개인을 의미합니다."},
        "metadata": {"contexts": "제1조의2(정의) “거주자”란 국내에 주소를 두거나 183일 이상의 거소를 둔 개인을 말한다."},
    },
    {
        "inputs": {"question": "'비거주자'는 소득세법에 따라 어떻게 정의되나요?"},
        "outputs": {"answer": "'비거주자'는 거주자가 아닌 개인을 의미합니다."},
        "metadata": {"contexts": "제1조의2(정의) “비거주자”란 거주자가 아닌 개인을 말한다."},
    },
    {
        "inputs": {"question": "소득세법에 따른 '내국법인'은 누구를 의미하나요?"},
        "outputs": {"answer": "'내국법인'은 법인세법 제2조 제1호에 따른 내국법인을 의미합니다."},
        "metadata": {"contexts": "제1조의2(정의) “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다."},
    },
    {
        "inputs": {"question": "소득세법에 따라 소득세를 납부할 의무가 있는 사람은 누구인가요?"},
        "outputs": {"answer": "거주자 및 국내원천소득이 있는 비거주자는 소득세를 납부할 의무가 있습니다."},
        "metadata": {"contexts": "제2조(납세의무) 거주자 및 국내원천소득이 있는 비거주자는 소득세를 납부할 의무가 있다."},
    },
    {
        "inputs": {"question": "거주자의 과세 범위는 무엇인가요?"},
        "outputs": {"answer": "거주자는 법에서 규정한 모든 소득에 대해 과세되며, 비거주자는 국내원천소득에 대해서만 과세됩니다."},
        "metadata": {"contexts": "제3조(과세소득의 범위) 거주자는 법에서 규정한 모든 소득에 대해 과세되며, 비거주자는 국내원천소득에 대해서만 과세된다."},
    },
    {
        "inputs": {"question": "소득세법에 따라 소득은 어떻게 분류되나요?"},
        "outputs": {"answer": "소득은 종합소득, 퇴직소득, 양도소득으로 분류됩니다."},
        "metadata": {"contexts": "제4조(소득의 구분) 소득은 종합소득, 퇴직소득, 양도소득으로 분류된다."},
    },
    {
        "inputs": {"question": "종합소득이란 무엇인가요?"},
        "outputs": {"answer": "종합소득은 이자소득, 배당소득, 사업소득, 근로소득, 연금소득 및 기타소득을 포함합니다."},
        "metadata": {"contexts": "제4조(소득의 구분) 종합소득은 이자소득, 배당소득, 사업소득, 근로소득, 연금소득 및 기타소득을 포함한다."},
    },
    {
        "inputs": {"question": "세금이 면제되는 소득의 종류는 무엇인가요?"},
        "outputs": {"answer": "비과세 소득에는 공익신탁의 이익, 특정 사업소득 및 기타 법에서 정한 특정 소득이 포함됩니다."},
        "metadata": {"contexts": "제12조(비과세소득) 비과세 소득에는 공익신탁의 이익, 특정 사업소득 및 기타 법에서 정한 특정 소득이 포함된다."},
    },
    {
        "inputs": {"question": "소득세의 과세기간은 어떻게 되나요?"},
        "outputs": {"answer": "소득세의 과세기간은 매년 1월 1일부터 12월 31일까지입니다."},
        "metadata": {"contexts": "제5조(과세기간) 소득세의 과세기간은 매년 1월 1일부터 12월 31일까지이다."},
    },
    {
        "inputs": {"question": "거주자의 소득세 납세지는 어디인가요?"},
        "outputs": {"answer": "거주자의 소득세 납세지는 주소지이며, 주소지가 없으면 거소지입니다."},
        "metadata": {"contexts": "제6조(납세지) 거주자의 소득세 납세지는 주소지이며, 주소지가 없으면 거소지이다."},
    },
    {
        "inputs": {"question": "비거주자의 소득세 납세지는 어디인가요?"},
        "outputs": {"answer": "비거주자의 소득세 납세지는 국내사업장의 소재지입니다. 국내사업장이 여러 곳인 경우 주된 사업장의 소재지가 납세지가 됩니다."},
        "metadata": {"contexts": "제6조(납세지) 비거주자의 소득세 납세지는 국내사업장의 소재지이다. 국내사업장이 여러 곳인 경우 주된 사업장의 소재지이다."},
    },
    {
        "inputs": {"question": "납세지가 불분명한 경우 어떻게 되나요?"},
        "outputs": {"answer": "납세지가 불분명한 경우 대통령령으로 정합니다."},
        "metadata": {"contexts": "제6조(납세지) 납세지가 불분명한 경우에는 대통령령으로 정한다."},
    },
    {
        "inputs": {"question": "원천징수세액의 납세지는 어떻게 결정되나요?"},
        "outputs": {"answer": "원천징수세액의 납세지는 원천징수자의 종류와 위치에 따라 결정됩니다."},
        "metadata": {"contexts": "제7조(원천징수 등의 경우의 납세지) 원천징수세액의 납세지는 원천징수자의 종류와 위치에 따라 결정된다."},
    },
    {
        "inputs": {"question": "납세자의 사망 시 납세지는 어떻게 되나요?"},
        "outputs": {"answer": "납세자의 사망 시 상속인 또는 납세관리인의 주소지나 거소지가 납세지가 됩니다."},
        "metadata": {"contexts": "제8조(상속 등의 경우의 납세지) 납세자의 사망 시 상속인 또는 납세관리인의 주소지나 거소지가 납세지가 된다."},
    },
    {
        "inputs": {"question": "신탁 소득에 대한 납세의 범위는 무엇인가요?"},
        "outputs": {"answer": "신탁 소득에 대한 납세의 범위는 신탁의 수익자가 해당 소득에 대해 납세의무를 집니다."},
        "metadata": {"contexts": "제2조의3(신탁재산 귀속 소득에 대한 납세의무의 범위) 신탁 소득에 대한 납세의 범위는 신탁의 수익자가 해당 소득에 대해 납세의무를 진다."},
    },
    {
        "inputs": {"question": "원천징수 대상 소득은 무엇인가요?"},
        "outputs": {"answer": "이자소득, 배당소득 및 기타 법에서 정한 소득은 원천징수 대상입니다."},
        "metadata": {"contexts": "제14조(과세표준의 계산) 이자소득, 배당소득 및 기타 법에서 정한 소득은 원천징수 대상이다."},
    },
    {
        "inputs": {"question": "공동 소유 자산의 양도소득은 어떻게 과세되나요?"},
        "outputs": {"answer": "공동 소유 자산의 양도소득은 각 거주자 소유 지분에 따라 과세됩니다."},
        "metadata": {"contexts": "제14조(과세표준의 계산) 공동 소유 자산의 양도소득은 각 거주자 소유 지분에 따라 과세된다."},
    },
    {
        "inputs": {"question": "이자 소득의 출처는 무엇인가요?"},
        "outputs": {"answer": "이자 소득의 출처는 정부 및 지방자치단체가 발행한 채권, 법인이 발행한 채권, 국내외 은행 예금 등입니다."},
        "metadata": {"contexts": "제16조(이자소득) 이자 소득의 출처는 정부 및 지방자치단체가 발행한 채권, 법인이 발행한 채권, 국내외 은행 예금 등이다."},
    },
    {
        "inputs": {"question": "소득세법에서 배당소득은 어떻게 정의되나요?"},
        "outputs": {"answer": "배당소득은 국내외 법인으로부터 받는 배당금 및 배분금, 기타 법에서 정한 소득을 포함합니다."},
        "metadata": {"contexts": "제17조(배당소득) 배당소득은 국내외 법인으로부터 받는 배당금 및 배분금, 기타 법에서 정한 소득을 포함한다."}
    }
]

# Create the dataset and examples in LangSmith
# 수정된 부분: dataset_name 값
dataset_name = "income_tax_dataset"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    # NOTE: 강의에서 client.create_examples 내 작성한 examples 내용을 위로 분리함.
    examples=examples
)

# 6.evaluators

### 6.1 correctness

In [ ]:
from typing_extensions import Annotated, TypedDict

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. (2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
# 수정된 부분: grader_llm = ChatOpenAI(model="gpt-4.1", temperature=0).with_structured_output(
grader_llm = ChatUpstage(model="solar-pro3").with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

### 6.2. Relevance

In [ ]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool, ..., "Provide the score on whether the answer addresses the question"
    ]

# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
# 수정된 부분: relevance_llm = ChatOpenAI(model="gpt-4.1", temperature=0).with_structured_output(
relevance_llm = ChatUpstage(model="solar-pro3").with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### 6.4. Retrieval relevance

In [ ]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise",
    ]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a set of FACTS provided by the student. Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
# 수정된 부분: retrieval_relevance_llm = ChatOpenAI(model="gpt-4.1", temperature=0).with_structured_output(
retrieval_relevance_llm = ChatUpstage(model="solar-pro3").with_structured_output(
	RetrievalRelevanceGrade, method="json_schema", strict=True
)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### 7. 평가 실행

In [ ]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    # NOTE: Evaluation에 사용될 dataset의 이름
    data=dataset_name,
    # NOTE: 실행할 Evaluator의 종류
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    # 수정된 부분: experiment_prefix="rag-doc-relevance",
    experiment_prefix="inflearn-correctness-relevance-groundedness-Retrieval_relevance",
    # 수정된 부분: metadata={"version": "LCEL context, gpt-4-0125-preview"},
    metadata={"version": "income tax v1, solar-pro3"},
)

# Explore results locally as a dataframe if you have pandas installed
# experiment_results.to_pandas()